# Notebook 06 — Fold-Local Average Uniqueness and Optuna Model Selection

## Purpose

Tune Random Forest and XGBoost on the frozen Stage 05 v2 purged anchored
walk-forward folds using the Stage 04 v8 35-feature schema.

### Training sample weights

- Compute Average Uniqueness independently inside every fold.
- Use only the current fold's retained training events.
- Compute concurrency within each symbol.
- Apply weights to Logistic Regression, Random Forest, and XGBoost.
- Keep the Dummy prior baseline unweighted.
- Keep validation metrics unweighted.

### Optuna

- 30 COMPLETE trials for Random Forest.
- 30 COMPLETE trials for XGBoost.
- No pruning; every trial evaluates all five folds.
- Primary objective: equal-fold mean ROC AUC.

### Frozen hierarchical selection

1. Find the best mean fold ROC AUC.
2. Retain trials within 0.005 of that mean.
3. Maximize worst-fold ROC AUC.
4. Minimize fold ROC AUC standard deviation.
5. Use lower trial number as final tie-break.

Threshold `0.50` is diagnostic only. The unseen test is not opened.


In [ ]:
from __future__ import annotations

from pathlib import Path
import gc
import json
import sys
import time

import numpy as np
import optuna
import pandas as pd
import yaml

print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("optuna:", optuna.__version__)


## 1. Locate repository and import project modules

In [ ]:
def locate_repository_root(start: Path) -> Path:
    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / "configs").exists()
            and (candidate / "notebooks").exists()
            and (candidate / "src").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Repository root was not found. "
        "Open the project root in VS Code."
    )


REPOSITORY_ROOT = locate_repository_root(Path.cwd())

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.evaluation.classification_metrics import (
    binary_classification_metrics,
)
from src.models.baselines import (
    build_dummy_prior_pipeline,
    build_logistic_regression_pipeline,
)
from src.models.random_forest import (
    build_random_forest_pipeline,
    sample_random_forest_params,
)
from src.models.tuning import (
    aggregate_model_fold_metrics,
    feature_columns_from_manifest,
    fold_auc_summary,
    hierarchical_near_best_ranking,
    select_optuna_trial_by_hierarchy,
)
from src.models.xgboost_model import (
    build_xgboost_pipeline,
    sample_xgboost_params,
)
from src.utils.paths import (
    data_paths,
    repository_result_paths,
    resolve_data_root,
)
from src.utils.reproducibility import (
    git_commit_sha,
    software_manifest,
    stable_object_hash,
)
from src.validation.folds import WalkForwardFold
from src.validation.purged_walk_forward import (
    fold_membership_masks,
    normalize_candidate_event_panel,
)
from src.validation.sample_weighting import (
    AVERAGE_UNIQUENESS_SCHEMA_VERSION,
    compute_fold_train_average_uniqueness,
    summarize_fold_average_uniqueness,
)

print("Repository root:", REPOSITORY_ROOT)


## 2. Load frozen Stage 04/05 artifacts and Stage 06 v3 configuration

In [ ]:
def load_yaml(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = yaml.safe_load(file_obj)

    if not isinstance(value, dict):
        raise ValueError(
            f"Configuration must be a mapping: {path}"
        )

    return value


paths_config = load_yaml(
    REPOSITORY_ROOT / "configs" / "paths.yaml"
)
optuna_config = load_yaml(
    REPOSITORY_ROOT / "configs" / "optuna.yaml"
)
models_config = load_yaml(
    REPOSITORY_ROOT / "configs" / "models.yaml"
)

DATA_ROOT = resolve_data_root(
    paths_config,
    REPOSITORY_ROOT,
)
DATA_PATHS = data_paths(
    DATA_ROOT,
    paths_config,
)
RESULT_PATHS = repository_result_paths(
    REPOSITORY_ROOT,
    paths_config,
)

CANDIDATE_TRAIN_PATH = (
    DATA_PATHS["candidate_data"] / "train"
)
LABELED_TRAIN_PATH = (
    DATA_PATHS["labeled_data"] / "train"
)

feature_manifest_path = (
    RESULT_PATHS["manifests"]
    / "04_approved_model_features.csv"
)
stage04_policy_path = (
    RESULT_PATHS["manifests"]
    / "04_candidate_filter_policy.json"
)
stage05_manifest_path = (
    RESULT_PATHS["manifests"]
    / "05_purged_anchored_walk_forward_manifest.json"
)
fold_boundaries_path = (
    RESULT_PATHS["folds"]
    / "05_purged_anchored_walk_forward_boundaries.csv"
)
validation_assignments_path = (
    RESULT_PATHS["folds"]
    / "05_validation_event_assignments.csv"
)

for required_path in [
    feature_manifest_path,
    stage04_policy_path,
    stage05_manifest_path,
    fold_boundaries_path,
    validation_assignments_path,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            "Required frozen-stage artifact is missing: "
            f"{required_path}"
        )

feature_manifest_df = pd.read_csv(
    feature_manifest_path,
    low_memory=False,
)
stage04_policy = json.loads(
    stage04_policy_path.read_text(encoding="utf-8")
)
stage05_manifest = json.loads(
    stage05_manifest_path.read_text(encoding="utf-8")
)
fold_boundaries_df = pd.read_csv(
    fold_boundaries_path,
    low_memory=False,
)
validation_assignments_df = pd.read_csv(
    validation_assignments_path,
    low_memory=False,
)

(
    MODEL_FEATURES,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
) = feature_columns_from_manifest(
    feature_manifest_df
)

SEED = int(
    optuna_config["reproducibility"]["random_seed"]
)
N_COMPLETE_TRIALS = int(
    optuna_config["optimization"][
        "required_complete_trials_per_model"
    ]
)
MEAN_AUC_TOLERANCE = float(
    optuna_config["hierarchical_trial_selection"][
        "near_best_mean_auc_tolerance"
    ]
)
DIAGNOSTIC_THRESHOLD = float(
    optuna_config["objective"][
        "probability_threshold_for_diagnostics"
    ]
)

WEIGHTED_MODELS = set(
    models_config["sample_weighting"]["applied_models"]
)

assert optuna_config["status"] == (
    "stage_06_configured_v3"
)
assert models_config["status"] == (
    "stage_06_configured_v3"
)
assert stage05_manifest["status"] == (
    "frozen_after_integrity_checks"
)
assert stage05_manifest["unseen_test_used"] is False
assert stage05_manifest[
    "fold_design_schema_version"
] == (
    "stage05_v2_candidate_event_count_balanced_contiguous"
)
assert stage04_policy["primary_side"] == "long"
assert np.isclose(
    float(
        stage04_policy[
            "primary_threshold_fraction"
        ]
    ),
    0.15,
)

assert len(MODEL_FEATURES) == 35
assert len(NUMERIC_FEATURES) == 34
assert CATEGORICAL_FEATURES == ["gmma_state"]
assert N_COMPLETE_TRIALS == 30
assert np.isclose(MEAN_AUC_TOLERANCE, 0.005)

assert AVERAGE_UNIQUENESS_SCHEMA_VERSION == (
    models_config["sample_weighting"][
        "schema_version"
    ]
)
assert models_config["sample_weighting"][
    "validation_events_used_to_compute_weights"
] is False
assert models_config["sample_weighting"][
    "validation_metrics_weighted"
] is False
assert models_config["sample_weighting"][
    "dummy_prior_weighted"
] is False
assert WEIGHTED_MODELS == {
    "logistic_regression",
    "random_forest",
    "xgboost",
}

assert optuna_config["pruner"]["name"] == (
    "NopPruner"
)
assert optuna_config["pruner"][
    "pruning_enabled"
] is False
assert optuna_config["optimization"][
    "every_trial_evaluates_all_folds"
] is True
assert optuna_config["objective"]["formula"] == (
    "mean_fold_roc_auc"
)

assert int(
    models_config["random_forest"][
        "search_space"
    ]["n_estimators"]["high"]
) == 150
assert int(
    models_config["random_forest"][
        "search_space"
    ]["max_depth"]["high"]
) == 15
assert int(
    models_config["xgboost"][
        "search_space"
    ]["n_estimators"]["high"]
) == 150
assert int(
    models_config["xgboost"][
        "search_space"
    ]["max_depth"]["high"]
) == 15

print("Model features:", len(MODEL_FEATURES))
print("Numeric features:", len(NUMERIC_FEATURES))
print("Categorical features:", CATEGORICAL_FEATURES)
print(
    "Required COMPLETE trials per tuned model:",
    N_COMPLETE_TRIALS,
)
print("Pruning enabled: False")
print("Mean-AUC near-best tolerance:", MEAN_AUC_TOLERANCE)
print(
    "Average Uniqueness schema:",
    AVERAGE_UNIQUENESS_SCHEMA_VERSION,
)
print("Unseen-test used: False")


## 3. Reconstruct the exact frozen Stage 05 v2 folds

In [ ]:
def boundary_row_to_fold(row) -> WalkForwardFold:
    return WalkForwardFold(
        fold_id=int(row.fold_id),
        calendar_start_date=pd.Timestamp(
            row.calendar_start_date
        ),
        train_end_date=pd.Timestamp(
            row.train_end_date
        ),
        embargo_start_date=pd.Timestamp(
            row.embargo_start_date
        ),
        embargo_end_date=pd.Timestamp(
            row.embargo_end_date
        ),
        validation_start_date=pd.Timestamp(
            row.validation_start_date
        ),
        validation_end_date=pd.Timestamp(
            row.validation_end_date
        ),
        train_end_calendar_index=int(
            row.train_end_calendar_index
        ),
        validation_start_calendar_index=int(
            row.validation_start_calendar_index
        ),
        validation_end_calendar_index=int(
            row.validation_end_calendar_index
        ),
        embargo_trading_days=int(
            row.embargo_trading_days
        ),
        validation_target_candidate_events=float(
            row.validation_target_candidate_events
        ),
        validation_candidate_events=int(
            row.validation_candidate_events
        ),
        validation_event_count_deviation=float(
            row.validation_event_count_deviation
        ),
        validation_event_count_relative_deviation=float(
            row.validation_event_count_relative_deviation
        ),
        validation_horizon_candidate_events=int(
            row.validation_horizon_candidate_events
        ),
        validation_partition_method=str(
            row.validation_partition_method
        ),
    )


folds = [
    boundary_row_to_fold(row)
    for row in fold_boundaries_df.itertuples(
        index=False
    )
]

assert len(folds) == 5
assert [
    fold.fold_id
    for fold in folds
] == [1, 2, 3, 4, 5]
assert all(
    fold.validation_partition_method
    == (
        "candidate_event_count_balanced_"
        "contiguous_calendar"
    )
    for fold in folds
)

display(fold_boundaries_df)


## 4. Build the pooled Stage 04 v8 candidate-event panel

In [ ]:
candidate_files = sorted(
    CANDIDATE_TRAIN_PATH.glob(
        "*_train_candidates.csv"
    )
)

if len(candidate_files) != int(
    stage05_manifest[
        "candidate_population"
    ]["symbols"]
):
    raise AssertionError(
        "Candidate-file count no longer matches Stage 05."
    )


def candidate_symbol_from_path(path: Path) -> str:
    suffix = "_train_candidates.csv"

    if not path.name.endswith(suffix):
        raise ValueError(
            f"Unexpected candidate file: {path.name}"
        )

    return path.name[:-len(suffix)]


candidate_parts = []
candidate_error_rows = []
started = time.perf_counter()

usecols = [
    "dEven",
    "event_end_date",
    "meta_label",
    *MODEL_FEATURES,
]

for file_number, path in enumerate(
    candidate_files,
    start=1,
):
    symbol = candidate_symbol_from_path(path)

    try:
        frame = pd.read_csv(
            path,
            usecols=usecols,
            low_memory=False,
        )
        frame.insert(0, "symbol", symbol)

        event_dates = pd.to_datetime(
            frame["dEven"],
            errors="coerce",
        )
        frame.insert(
            0,
            "event_id",
            (
                symbol
                + "::"
                + event_dates.dt.strftime("%Y-%m-%d")
            ),
        )
        candidate_parts.append(frame)

    except Exception as exc:
        candidate_error_rows.append(
            {
                "symbol": symbol,
                "file_name": path.name,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

    if (
        file_number == 1
        or file_number % 50 == 0
        or file_number == len(candidate_files)
    ):
        elapsed = time.perf_counter() - started
        print(
            f"[candidate panel] "
            f"[{file_number:>4}/{len(candidate_files)}] "
            f"elapsed={elapsed:,.1f}s "
            f"errors={len(candidate_error_rows)}"
        )

candidate_errors_df = pd.DataFrame(
    candidate_error_rows,
    columns=[
        "symbol",
        "file_name",
        "error_type",
        "error_message",
    ],
)

candidate_errors_df.to_csv(
    RESULT_PATHS["audits"]
    / "06_candidate_panel_errors.csv",
    index=False,
    encoding="utf-8-sig",
)

if not candidate_errors_df.empty:
    raise RuntimeError(
        "Candidate panel contains loading errors. "
        "Review 06_candidate_panel_errors.csv"
    )

candidate_panel = normalize_candidate_event_panel(
    pd.concat(
        candidate_parts,
        ignore_index=True,
    )
)

for feature in NUMERIC_FEATURES:
    candidate_panel[feature] = pd.to_numeric(
        candidate_panel[feature],
        errors="coerce",
    )

candidate_panel["gmma_state"] = (
    candidate_panel[
        "gmma_state"
    ].astype("string")
)

expected_events = int(
    stage05_manifest[
        "candidate_population"
    ]["events"]
)

assert len(candidate_panel) == expected_events
assert candidate_panel["event_id"].is_unique
assert candidate_panel["meta_label"].isin(
    [0, 1]
).all()

numeric_array = candidate_panel[
    NUMERIC_FEATURES
].to_numpy(dtype=float)

assert not np.isinf(numeric_array).any()

panel_summary_df = pd.DataFrame(
    [
        {
            "candidate_events": len(candidate_panel),
            "symbols": int(
                candidate_panel[
                    "symbol"
                ].nunique()
            ),
            "features": len(MODEL_FEATURES),
            "numeric_features": len(
                NUMERIC_FEATURES
            ),
            "categorical_features": len(
                CATEGORICAL_FEATURES
            ),
            "positive_fraction": float(
                candidate_panel[
                    "meta_label"
                ].mean()
            ),
            "numeric_missing_values": int(
                candidate_panel[
                    NUMERIC_FEATURES
                ].isna().sum().sum()
            ),
            "categorical_missing_values": int(
                candidate_panel[
                    CATEGORICAL_FEATURES
                ].isna().sum().sum()
            ),
            "duplicate_event_ids": int(
                candidate_panel[
                    "event_id"
                ].duplicated().sum()
            ),
            "unseen_test_used": False,
        }
    ]
)

panel_summary_df.to_csv(
    RESULT_PATHS["audits"]
    / "06_candidate_panel_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

display(panel_summary_df)


## 5. Reproduce frozen Stage 05 validation membership exactly

In [ ]:
validation_assignments_df["event_id"] = (
    validation_assignments_df[
        "event_id"
    ].astype(str)
)
validation_assignments_df["fold_id"] = pd.to_numeric(
    validation_assignments_df["fold_id"],
    errors="raise",
).astype(int)

fold_reproduction_rows = []
fold_position_cache = {}

for fold in folds:
    masks = fold_membership_masks(
        candidate_panel,
        fold,
    )

    train_positions = np.flatnonzero(
        masks.train.to_numpy(dtype=bool)
    )
    validation_positions = np.flatnonzero(
        masks.validation.to_numpy(dtype=bool)
    )

    fold_position_cache[fold.fold_id] = {
        "train": train_positions,
        "validation": validation_positions,
    }

    reproduced_ids = set(
        candidate_panel.iloc[
            validation_positions
        ]["event_id"].astype(str)
    )
    frozen_ids = set(
        validation_assignments_df.loc[
            validation_assignments_df[
                "fold_id"
            ].eq(fold.fold_id),
            "event_id",
        ].astype(str)
    )

    missing_from_reproduction = (
        frozen_ids - reproduced_ids
    )
    unexpected_in_reproduction = (
        reproduced_ids - frozen_ids
    )

    fold_reproduction_rows.append(
        {
            "fold_id": fold.fold_id,
            "frozen_validation_events": len(
                frozen_ids
            ),
            "reproduced_validation_events": len(
                reproduced_ids
            ),
            "reproduced_train_events": int(
                len(train_positions)
            ),
            "missing_from_reproduction": len(
                missing_from_reproduction
            ),
            "unexpected_in_reproduction": len(
                unexpected_in_reproduction
            ),
            "exact_membership_match": (
                not missing_from_reproduction
                and not unexpected_in_reproduction
            ),
        }
    )

fold_reproduction_audit_df = pd.DataFrame(
    fold_reproduction_rows
)

fold_reproduction_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "06_fold_reproduction_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

assert fold_reproduction_audit_df[
    "exact_membership_match"
].all()

display(fold_reproduction_audit_df)


## 6. Load symbol-specific labeled-train calendars

Average Uniqueness is evaluated over each security's own sequence of trading
observations. Different securities never contribute to one another's
concurrency.


In [ ]:
def labeled_symbol_from_path(path: Path) -> str:
    suffix = "_train_labeled.csv"

    if not path.name.endswith(suffix):
        raise ValueError(
            f"Unexpected labeled-train file: {path.name}"
        )

    return path.name[:-len(suffix)]


labeled_train_files = sorted(
    LABELED_TRAIN_PATH.glob(
        "*_train_labeled.csv"
    )
)

calendar_error_rows = []
symbol_calendars = {}
calendar_audit_rows = []

started = time.perf_counter()

for file_number, path in enumerate(
    labeled_train_files,
    start=1,
):
    symbol = labeled_symbol_from_path(path)

    try:
        date_frame = pd.read_csv(
            path,
            usecols=["dEven"],
            low_memory=False,
        )

        calendar = pd.DatetimeIndex(
            sorted(
                pd.to_datetime(
                    date_frame["dEven"],
                    errors="coerce",
                )
                .dropna()
                .dt.normalize()
                .unique()
            )
        )

        if calendar.empty:
            raise ValueError(
                "Labeled-train calendar is empty."
            )

        if calendar.has_duplicates:
            raise AssertionError(
                "Labeled-train calendar has duplicates."
            )

        symbol_calendars[symbol] = calendar

        calendar_audit_rows.append(
            {
                "symbol": symbol,
                "calendar_observations": int(
                    len(calendar)
                ),
                "first_date": calendar.min(),
                "last_date": calendar.max(),
                "invalid_date_rows": int(
                    pd.to_datetime(
                        date_frame["dEven"],
                        errors="coerce",
                    ).isna().sum()
                ),
            }
        )

    except Exception as exc:
        calendar_error_rows.append(
            {
                "symbol": symbol,
                "file_name": path.name,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

    if (
        file_number == 1
        or file_number % 50 == 0
        or file_number == len(
            labeled_train_files
        )
    ):
        elapsed = time.perf_counter() - started
        print(
            f"[symbol calendars] "
            f"[{file_number:>4}/"
            f"{len(labeled_train_files)}] "
            f"elapsed={elapsed:,.1f}s "
            f"errors={len(calendar_error_rows)}"
        )

calendar_errors_df = pd.DataFrame(
    calendar_error_rows,
    columns=[
        "symbol",
        "file_name",
        "error_type",
        "error_message",
    ],
)
calendar_audit_df = pd.DataFrame(
    calendar_audit_rows
)

calendar_errors_df.to_csv(
    RESULT_PATHS["audits"]
    / "06_symbol_calendar_errors.csv",
    index=False,
    encoding="utf-8-sig",
)
calendar_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "06_symbol_calendar_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

if not calendar_errors_df.empty:
    raise RuntimeError(
        "Symbol calendar errors exist. "
        "Review 06_symbol_calendar_errors.csv"
    )

candidate_symbols = set(
    candidate_panel["symbol"].astype(str)
)
calendar_symbols = set(symbol_calendars)

assert candidate_symbols == calendar_symbols
assert len(symbol_calendars) == 499

display(calendar_audit_df.describe(include="all"))


## 7. Compute Fold-Local Average Uniqueness weights

For each fold, only retained training events are supplied to the concurrency
calculation. Validation events are excluded before weighting begins.

The event interval is inclusive of both `dEven` and `event_end_date`. Raw
uniqueness is normalized to mean one inside the current fold's training set.


In [ ]:
fold_training_weight_cache = {}
weight_event_parts = []
weight_symbol_audit_parts = []
weight_summary_rows = []

for fold in folds:
    train_positions = fold_position_cache[
        fold.fold_id
    ]["train"]
    validation_positions = fold_position_cache[
        fold.fold_id
    ]["validation"]

    train_events = candidate_panel.iloc[
        train_positions
    ][
        [
            "event_id",
            "symbol",
            "dEven",
            "event_end_date",
        ]
    ].copy()

    validation_event_ids = set(
        candidate_panel.iloc[
            validation_positions
        ]["event_id"].astype(str)
    )

    fold_weights, symbol_weight_audit = (
        compute_fold_train_average_uniqueness(
            train_events,
            symbol_calendars,
        )
    )

    weight_event_ids = set(
        fold_weights["event_id"].astype(str)
    )

    validation_ids_used = len(
        weight_event_ids.intersection(
            validation_event_ids
        )
    )

    if validation_ids_used != 0:
        raise AssertionError(
            "Validation events entered fold-training "
            "weight construction."
        )

    if weight_event_ids != set(
        train_events["event_id"].astype(str)
    ):
        raise AssertionError(
            "Average Uniqueness output does not exactly "
            "cover the fold-training events."
        )

    weight_by_event_id = (
        fold_weights.set_index(
            "event_id"
        )["sample_weight"]
    )

    aligned_weight = (
        train_events["event_id"]
        .astype(str)
        .map(weight_by_event_id)
        .to_numpy(dtype=float)
    )

    if not np.isfinite(aligned_weight).all():
        raise AssertionError(
            "Aligned fold-training weights are non-finite."
        )

    if not np.isclose(
        float(aligned_weight.mean()),
        1.0,
        atol=1.0e-12,
        rtol=1.0e-12,
    ):
        raise AssertionError(
            "Aligned fold-training weights do not "
            "average to one."
        )

    fold_training_weight_cache[
        fold.fold_id
    ] = aligned_weight

    fold_weights.insert(
        0,
        "fold_id",
        int(fold.fold_id),
    )
    weight_event_parts.append(fold_weights)

    symbol_weight_audit.insert(
        0,
        "fold_id",
        int(fold.fold_id),
    )
    weight_symbol_audit_parts.append(
        symbol_weight_audit
    )

    summary = summarize_fold_average_uniqueness(
        fold_weights,
        fold_id=fold.fold_id,
        validation_events_used=(
            validation_ids_used
        ),
    )
    summary.update(
        {
            "train_end_date": fold.train_end_date,
            "validation_start_date": (
                fold.validation_start_date
            ),
            "validation_end_date": (
                fold.validation_end_date
            ),
            "validation_events": int(
                len(validation_positions)
            ),
        }
    )
    weight_summary_rows.append(summary)

    print(
        f"fold={fold.fold_id} "
        f"train_events={len(train_events):,} "
        f"mean_raw_uniqueness="
        f"{summary['mean_raw_average_uniqueness']:.6f} "
        f"effective_sample_fraction="
        f"{summary['effective_sample_fraction']:.6f} "
        f"validation_events_used=0"
    )

all_fold_train_weights_df = pd.concat(
    weight_event_parts,
    ignore_index=True,
)
average_uniqueness_symbol_audit_df = pd.concat(
    weight_symbol_audit_parts,
    ignore_index=True,
)
average_uniqueness_audit_df = pd.DataFrame(
    weight_summary_rows
)

all_fold_train_weights_df.to_csv(
    RESULT_PATHS["folds"]
    / "06_fold_train_average_uniqueness_weights.csv",
    index=False,
    encoding="utf-8-sig",
)

average_uniqueness_symbol_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "06_average_uniqueness_symbol_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

average_uniqueness_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "06_average_uniqueness_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

display(average_uniqueness_audit_df)


## 8. Frozen fold evaluation

Preprocessing and all sample weighting are fold-local.

- Logistic Regression, Random Forest, and XGBoost receive Average Uniqueness
  through `model__sample_weight`.
- Dummy prior remains unweighted.
- XGBoost's optional class-ratio mode uses weighted class mass.
- Validation metrics are always calculated without sample weights.


In [ ]:
def xgb_scale_pos_weight(
    y_train: np.ndarray,
    sample_weight: np.ndarray,
    mode: str,
) -> float:
    if mode == "none":
        return 1.0

    if mode != "fold_weighted_ratio":
        raise ValueError(
            f"Unknown class_weight_mode: {mode}"
        )

    y_array = np.asarray(
        y_train,
        dtype=int,
    )
    weight_array = np.asarray(
        sample_weight,
        dtype=float,
    )

    positive_mass = float(
        weight_array[y_array == 1].sum()
    )
    negative_mass = float(
        weight_array[y_array == 0].sum()
    )

    if positive_mass <= 0.0 or negative_mass <= 0.0:
        raise ValueError(
            "Both weighted classes are required in "
            "fold training data."
        )

    return negative_mass / positive_mass


def fit_predict_fold(
    *,
    model_name: str,
    fold: WalkForwardFold,
    params: dict | None = None,
    return_predictions: bool = True,
) -> tuple[dict, pd.DataFrame | None]:
    positions = fold_position_cache[
        fold.fold_id
    ]

    train = candidate_panel.iloc[
        positions["train"]
    ]
    validation = candidate_panel.iloc[
        positions["validation"]
    ]

    X_train = train[MODEL_FEATURES]
    y_train = train[
        "meta_label"
    ].to_numpy(dtype=int)

    X_validation = validation[
        MODEL_FEATURES
    ]
    y_validation = validation[
        "meta_label"
    ].to_numpy(dtype=int)

    train_weight = fold_training_weight_cache[
        fold.fold_id
    ]

    if len(train_weight) != len(train):
        raise AssertionError(
            "Fold-training sample-weight length mismatch."
        )

    fold_seed = SEED + int(fold.fold_id)

    if model_name == "dummy_prior":
        model = build_dummy_prior_pipeline(
            NUMERIC_FEATURES,
            CATEGORICAL_FEATURES,
            seed=fold_seed,
        )

    elif model_name == "logistic_regression":
        model = build_logistic_regression_pipeline(
            NUMERIC_FEATURES,
            CATEGORICAL_FEATURES,
            seed=fold_seed,
        )

    elif model_name == "random_forest":
        if params is None:
            raise ValueError(
                "Random Forest params are required."
            )

        model = build_random_forest_pipeline(
            NUMERIC_FEATURES,
            CATEGORICAL_FEATURES,
            params=params,
            seed=fold_seed,
        )

    elif model_name == "xgboost":
        if params is None:
            raise ValueError(
                "XGBoost params are required."
            )

        scale_pos_weight = xgb_scale_pos_weight(
            y_train,
            train_weight,
            str(
                params.get(
                    "class_weight_mode",
                    "none",
                )
            ),
        )

        model = build_xgboost_pipeline(
            NUMERIC_FEATURES,
            CATEGORICAL_FEATURES,
            params=params,
            seed=fold_seed,
            scale_pos_weight=scale_pos_weight,
        )

    else:
        raise ValueError(
            f"Unknown model_name: {model_name}"
        )

    fit_started = time.perf_counter()

    if model_name in WEIGHTED_MODELS:
        model.fit(
            X_train,
            y_train,
            model__sample_weight=train_weight,
        )
        sample_weighting_applied = True
    else:
        model.fit(
            X_train,
            y_train,
        )
        sample_weighting_applied = False

    fit_seconds = (
        time.perf_counter() - fit_started
    )

    inference_started = time.perf_counter()
    probability = model.predict_proba(
        X_validation
    )[:, 1]
    inference_seconds = (
        time.perf_counter() - inference_started
    )

    metrics = binary_classification_metrics(
        y_validation,
        probability,
        threshold=DIAGNOSTIC_THRESHOLD,
    )
    metrics.update(
        {
            "model_name": model_name,
            "fold_id": int(fold.fold_id),
            "train_events": int(len(train)),
            "validation_events": int(
                len(validation)
            ),
            "train_positive_fraction": float(
                y_train.mean()
            ),
            "validation_positive_fraction": float(
                y_validation.mean()
            ),
            "average_uniqueness_weighting_applied": (
                sample_weighting_applied
            ),
            "train_sample_weight_mean": (
                float(train_weight.mean())
                if sample_weighting_applied
                else 1.0
            ),
            "train_sample_weight_min": (
                float(train_weight.min())
                if sample_weighting_applied
                else 1.0
            ),
            "train_sample_weight_max": (
                float(train_weight.max())
                if sample_weighting_applied
                else 1.0
            ),
            "validation_metrics_weighted": False,
            "fit_seconds": float(
                fit_seconds
            ),
            "inference_seconds": float(
                inference_seconds
            ),
        }
    )

    predictions = None

    if return_predictions:
        predictions = validation[
            [
                "event_id",
                "symbol",
                "dEven",
                "event_end_date",
                "meta_label",
            ]
        ].copy()
        predictions.insert(
            0,
            "fold_id",
            int(fold.fold_id),
        )
        predictions.insert(
            0,
            "model_name",
            model_name,
        )
        predictions[
            "probability_positive"
        ] = probability
        predictions[
            "diagnostic_prediction_at_0_50"
        ] = (
            probability
            >= DIAGNOSTIC_THRESHOLD
        ).astype(int)
        predictions[
            "validation_metric_weight"
        ] = 1.0

    del model
    gc.collect()

    return metrics, predictions


## 9. Evaluate fixed baselines

In [ ]:
baseline_fold_metric_rows = []
baseline_prediction_parts = []

for model_name in [
    "dummy_prior",
    "logistic_regression",
]:
    print(f"\nBaseline: {model_name}")

    for fold in folds:
        metrics, predictions = fit_predict_fold(
            model_name=model_name,
            fold=fold,
            return_predictions=True,
        )

        baseline_fold_metric_rows.append(
            metrics
        )
        baseline_prediction_parts.append(
            predictions
        )

        print(
            f"  fold={fold.fold_id} "
            f"roc_auc={metrics['roc_auc']:.6f} "
            f"specificity@0.50="
            f"{metrics['specificity']:.6f} "
            f"sensitivity@0.50="
            f"{metrics['sensitivity']:.6f} "
            f"weighted="
            f"{metrics['average_uniqueness_weighting_applied']}"
        )

baseline_fold_metrics_df = pd.DataFrame(
    baseline_fold_metric_rows
)
baseline_predictions_df = pd.concat(
    baseline_prediction_parts,
    ignore_index=True,
)

display(baseline_fold_metrics_df)


## 10. Create versioned Optuna studies

Random Forest and XGBoost receive separate sampler instances. `NopPruner`
guarantees that every COMPLETE trial evaluates all five folds.


In [ ]:
study_signature_payload = {
    "feature_manifest": (
        feature_manifest_df.to_dict(
            orient="records"
        )
    ),
    "fold_boundaries": (
        fold_boundaries_df.to_dict(
            orient="records"
        )
    ),
    "optuna_config": optuna_config,
    "models_config": models_config,
    "stage04_policy": stage04_policy,
    "stage05_configuration_hash": (
        stage05_manifest[
            "configuration_hash"
        ]
    ),
    "average_uniqueness_schema": (
        AVERAGE_UNIQUENESS_SCHEMA_VERSION
    ),
    "average_uniqueness_audit": (
        average_uniqueness_audit_df.to_dict(
            orient="records"
        )
    ),
}

STUDY_SIGNATURE = stable_object_hash(
    study_signature_payload
)[:12]

database_path = (
    REPOSITORY_ROOT
    / optuna_config[
        "storage"
    ]["database_file"]
)
database_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

storage_url = (
    "sqlite:///"
    + database_path.resolve().as_posix()
)


def build_study_sampler():
    return optuna.samplers.TPESampler(
        seed=SEED,
        multivariate=bool(
            optuna_config[
                "sampler"
            ]["multivariate"]
        ),
    )


def build_study_pruner():
    return optuna.pruners.NopPruner()


study_names = {
    "random_forest": (
        "stage06_v3_random_forest_"
        f"{STUDY_SIGNATURE}"
    ),
    "xgboost": (
        "stage06_v3_xgboost_"
        f"{STUDY_SIGNATURE}"
    ),
}

print("Study signature:", STUDY_SIGNATURE)
print("Optuna database:", database_path)
print("Study names:", study_names)


## 11. Define the complete five-fold objective

The objective returned to Optuna is equal-fold mean ROC AUC. Worst-fold ROC AUC
and fold standard deviation are stored for the frozen post-search hierarchy.


In [ ]:
def create_objective(model_name: str):
    if model_name not in {
        "random_forest",
        "xgboost",
    }:
        raise ValueError(
            "Only Random Forest and XGBoost are tuned."
        )

    def objective(
        trial: optuna.Trial,
    ) -> float:
        params = (
            sample_random_forest_params(
                trial
            )
            if model_name == "random_forest"
            else sample_xgboost_params(
                trial
            )
        )

        fold_metric_rows = []

        for fold in folds:
            metrics, _ = fit_predict_fold(
                model_name=model_name,
                fold=fold,
                params=params,
                return_predictions=False,
            )
            fold_metric_rows.append(metrics)

        if len(fold_metric_rows) != len(folds):
            raise AssertionError(
                "A trial did not evaluate all frozen folds."
            )

        auc_summary = fold_auc_summary(
            [
                row["roc_auc"]
                for row in fold_metric_rows
            ]
        )

        trial.set_user_attr(
            "fold_metrics",
            fold_metric_rows,
        )
        trial.set_user_attr(
            "folds_evaluated",
            len(fold_metric_rows),
        )
        trial.set_user_attr(
            "mean_roc_auc",
            auc_summary["mean"],
        )
        trial.set_user_attr(
            "std_roc_auc",
            auc_summary["std"],
        )
        trial.set_user_attr(
            "min_roc_auc",
            auc_summary["minimum"],
        )
        trial.set_user_attr(
            "max_roc_auc",
            auc_summary["maximum"],
        )
        trial.set_user_attr(
            "mean_specificity_at_0_50",
            float(
                np.mean(
                    [
                        row["specificity"]
                        for row in fold_metric_rows
                    ]
                )
            ),
        )
        trial.set_user_attr(
            "average_uniqueness_weighting",
            True,
        )
        trial.set_user_attr(
            "validation_metrics_weighted",
            False,
        )

        return auc_summary["mean"]

    return objective


## 12. Run or resume 30 COMPLETE trials per tuned model

In [ ]:
def trial_state_counts(
    study: optuna.Study,
) -> dict[str, int]:
    return {
        state.name: sum(
            trial.state == state
            for trial in study.trials
        )
        for state in [
            optuna.trial.TrialState.COMPLETE,
            optuna.trial.TrialState.PRUNED,
            optuna.trial.TrialState.FAIL,
            optuna.trial.TrialState.RUNNING,
            optuna.trial.TrialState.WAITING,
        ]
    }


studies = {}

for model_name in [
    "random_forest",
    "xgboost",
]:
    study = optuna.create_study(
        study_name=study_names[model_name],
        storage=storage_url,
        direction="maximize",
        sampler=build_study_sampler(),
        pruner=build_study_pruner(),
        load_if_exists=True,
    )

    counts_before = trial_state_counts(
        study
    )
    complete_before = counts_before[
        "COMPLETE"
    ]
    remaining_complete_trials = max(
        0,
        N_COMPLETE_TRIALS - complete_before,
    )

    print(
        f"\n{model_name}: "
        f"complete_before={complete_before}, "
        f"remaining_complete_trials="
        f"{remaining_complete_trials}"
    )

    if remaining_complete_trials > 0:
        study.optimize(
            create_objective(model_name),
            n_trials=remaining_complete_trials,
            n_jobs=int(
                optuna_config[
                    "optimization"
                ]["n_jobs"]
            ),
            gc_after_trial=bool(
                optuna_config[
                    "optimization"
                ]["gc_after_trial"]
            ),
            show_progress_bar=True,
        )

    counts_after = trial_state_counts(
        study
    )

    if counts_after["COMPLETE"] < (
        N_COMPLETE_TRIALS
    ):
        raise RuntimeError(
            f"{model_name} has fewer than "
            f"{N_COMPLETE_TRIALS} COMPLETE trials."
        )

    if counts_after["PRUNED"] != 0:
        raise AssertionError(
            f"{model_name} contains pruned v3 trials "
            "despite NopPruner."
        )

    if counts_after["FAIL"] != 0:
        raise AssertionError(
            f"{model_name} contains failed v3 trials. "
            "Review the study before model selection."
        )

    studies[model_name] = study

    print(
        f"{model_name} trial states:",
        counts_after,
    )
    print(
        f"{model_name} best mean ROC AUC:",
        study.best_value,
    )


assert (
    studies["random_forest"].sampler
    is not studies["xgboost"].sampler
)
assert isinstance(
    studies["random_forest"].pruner,
    optuna.pruners.NopPruner,
)
assert isinstance(
    studies["xgboost"].pruner,
    optuna.pruners.NopPruner,
)

print("Distinct study sampler instances: True")
print("NopPruner used for both studies: True")


## 13. Select one configuration per model using the frozen hierarchy

In [ ]:
selected_trials = {}
trial_ranking_frames = []
trial_export_paths = {}

for model_name, study in studies.items():
    selected_trial, ranking = (
        select_optuna_trial_by_hierarchy(
            study,
            mean_tolerance=(
                MEAN_AUC_TOLERANCE
            ),
        )
    )

    selected_trials[model_name] = (
        selected_trial
    )

    ranking.insert(
        0,
        "model_name",
        model_name,
    )
    trial_ranking_frames.append(
        ranking
    )

    trials_df = study.trials_dataframe(
        attrs=(
            "number",
            "value",
            "datetime_start",
            "datetime_complete",
            "duration",
            "params",
            "user_attrs",
            "state",
        )
    )

    trials_df = trials_df.merge(
        ranking[
            [
                "trial_number",
                "inside_near_best_mean_auc_band",
                "hierarchical_rank",
                "selected_by_hierarchy",
            ]
        ],
        left_on="number",
        right_on="trial_number",
        how="left",
        validate="one_to_one",
    ).drop(columns=["trial_number"])

    trial_path = (
        RESULT_PATHS["optuna"]
        / f"06_{model_name}_trials.csv"
    )
    ranking_path = (
        RESULT_PATHS["optuna"]
        / (
            f"06_{model_name}_"
            "trial_selection_ranking.csv"
        )
    )

    trials_df.to_csv(
        trial_path,
        index=False,
        encoding="utf-8-sig",
    )
    ranking.to_csv(
        ranking_path,
        index=False,
        encoding="utf-8-sig",
    )

    trial_export_paths[model_name] = {
        "trials": trial_path,
        "ranking": ranking_path,
    }

    print(
        f"\n{model_name} selected trial:",
        selected_trial.number,
    )
    print(
        "  mean ROC AUC:",
        selected_trial.user_attrs[
            "mean_roc_auc"
        ],
    )
    print(
        "  worst-fold ROC AUC:",
        selected_trial.user_attrs[
            "min_roc_auc"
        ],
    )
    print(
        "  fold ROC AUC std:",
        selected_trial.user_attrs[
            "std_roc_auc"
        ],
    )
    print(
        "  params:",
        selected_trial.params,
    )

all_trial_rankings_df = pd.concat(
    trial_ranking_frames,
    ignore_index=True,
)

display(all_trial_rankings_df.loc[
    all_trial_rankings_df[
        "inside_near_best_mean_auc_band"
    ]
])


## 14. Evaluate selected tuned configurations on every frozen fold

This re-evaluation saves validation probabilities. Full-train fitting remains
Notebook 07.


In [ ]:
best_params = {
    model_name: dict(
        trial.params
    )
    for model_name, trial in (
        selected_trials.items()
    )
}

tuned_fold_metric_rows = []
tuned_prediction_parts = []

for model_name in [
    "random_forest",
    "xgboost",
]:
    print(
        f"\nSelected tuned model: "
        f"{model_name}"
    )

    for fold in folds:
        metrics, predictions = fit_predict_fold(
            model_name=model_name,
            fold=fold,
            params=best_params[
                model_name
            ],
            return_predictions=True,
        )

        tuned_fold_metric_rows.append(
            metrics
        )
        tuned_prediction_parts.append(
            predictions
        )

        print(
            f"  fold={fold.fold_id} "
            f"roc_auc={metrics['roc_auc']:.6f} "
            f"AP={metrics['average_precision']:.6f} "
            f"specificity@0.50="
            f"{metrics['specificity']:.6f} "
            f"sensitivity@0.50="
            f"{metrics['sensitivity']:.6f}"
        )

tuned_fold_metrics_df = pd.DataFrame(
    tuned_fold_metric_rows
)
tuned_predictions_df = pd.concat(
    tuned_prediction_parts,
    ignore_index=True,
)

all_fold_metrics_df = pd.concat(
    [
        baseline_fold_metrics_df,
        tuned_fold_metrics_df,
    ],
    ignore_index=True,
)

all_predictions_df = pd.concat(
    [
        baseline_predictions_df,
        tuned_predictions_df,
    ],
    ignore_index=True,
)

all_fold_metrics_df.to_csv(
    RESULT_PATHS["metrics"]
    / "06_model_selection_fold_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

all_predictions_df.to_csv(
    RESULT_PATHS["predictions"]
    / "06_walk_forward_validation_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

display(all_fold_metrics_df)


## 15. Equal-fold comparison and final tree-family decision

In [ ]:
model_selection_summary_df = (
    aggregate_model_fold_metrics(
        all_fold_metrics_df
    )
)

tree_summary = (
    model_selection_summary_df.loc[
        model_selection_summary_df[
            "model_name"
        ].isin(
            [
                "random_forest",
                "xgboost",
            ]
        )
    ].copy()
)

tree_model_ranking_df = (
    hierarchical_near_best_ranking(
        tree_summary,
        candidate_column="model_name",
        mean_column="mean_roc_auc",
        minimum_column="min_roc_auc",
        std_column="std_roc_auc",
        mean_tolerance=(
            MEAN_AUC_TOLERANCE
        ),
        numeric_final_tie_break=False,
    )
)

PRIMARY_SELECTED_MODEL = str(
    tree_model_ranking_df.loc[
        tree_model_ranking_df[
            "selected_by_hierarchy"
        ],
        "model_name",
    ].iloc[0]
)

CHALLENGER_MODEL = str(
    tree_model_ranking_df.loc[
        ~tree_model_ranking_df[
            "selected_by_hierarchy"
        ],
        "model_name",
    ].iloc[0]
)

selected_tree_row = (
    model_selection_summary_df.loc[
        model_selection_summary_df[
            "model_name"
        ].eq(
            PRIMARY_SELECTED_MODEL
        )
    ].iloc[0]
)

logistic_row = (
    model_selection_summary_df.loc[
        model_selection_summary_df[
            "model_name"
        ].eq(
            "logistic_regression"
        )
    ].iloc[0]
)

selection_decision = {
    "primary_selected_model": (
        PRIMARY_SELECTED_MODEL
    ),
    "challenger_model": (
        CHALLENGER_MODEL
    ),
    "selection_scope": [
        "random_forest",
        "xgboost",
    ],
    "optuna_objective": (
        "equal_fold_mean_roc_auc"
    ),
    "near_best_mean_auc_tolerance": (
        MEAN_AUC_TOLERANCE
    ),
    "hierarchical_rule": [
        "identify_best_mean_fold_roc_auc",
        (
            "retain_candidates_within_0.005_"
            "of_best_mean"
        ),
        "maximize_worst_fold_roc_auc",
        "minimize_fold_roc_auc_std",
        "deterministic_final_tie_break",
    ],
    "selected_trial_numbers": {
        model_name: int(
            trial.number
        )
        for model_name, trial in (
            selected_trials.items()
        )
    },
    "fold_weighting": "equal",
    "average_uniqueness_weighting": {
        "method": "average_uniqueness",
        "scope": (
            "current_fold_training_events_only"
        ),
        "concurrency": "within_symbol",
        "weighted_models": sorted(
            WEIGHTED_MODELS
        ),
        "dummy_prior_weighted": False,
        "validation_metrics_weighted": False,
    },
    "diagnostic_threshold": (
        DIAGNOSTIC_THRESHOLD
    ),
    "threshold_selected": False,
    "selected_tree_mean_roc_auc": float(
        selected_tree_row[
            "mean_roc_auc"
        ]
    ),
    "selected_tree_min_roc_auc": float(
        selected_tree_row[
            "min_roc_auc"
        ]
    ),
    "selected_tree_std_roc_auc": float(
        selected_tree_row[
            "std_roc_auc"
        ]
    ),
    "logistic_baseline_mean_roc_auc": float(
        logistic_row[
            "mean_roc_auc"
        ]
    ),
    "logistic_baseline_min_roc_auc": float(
        logistic_row[
            "min_roc_auc"
        ]
    ),
    "selected_tree_beats_logistic_on_mean_auc": bool(
        selected_tree_row[
            "mean_roc_auc"
        ]
        > logistic_row[
            "mean_roc_auc"
        ]
    ),
    "unseen_test_used": False,
}

model_selection_summary_df.to_csv(
    RESULT_PATHS["metrics"]
    / "06_model_selection_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

tree_model_ranking_df.to_csv(
    RESULT_PATHS["metrics"]
    / "06_tree_model_hierarchical_ranking.csv",
    index=False,
    encoding="utf-8-sig",
)

(
    RESULT_PATHS["manifests"]
    / "06_model_selection_decision.json"
).write_text(
    json.dumps(
        selection_decision,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

best_hyperparameters_payload = {
    "study_signature": STUDY_SIGNATURE,
    "random_seed": SEED,
    "selection_rule": (
        selection_decision[
            "hierarchical_rule"
        ]
    ),
    "near_best_mean_auc_tolerance": (
        MEAN_AUC_TOLERANCE
    ),
}

for model_name, trial in (
    selected_trials.items()
):
    best_hyperparameters_payload[
        model_name
    ] = {
        "selected_trial_number": int(
            trial.number
        ),
        "mean_roc_auc": float(
            trial.user_attrs[
                "mean_roc_auc"
            ]
        ),
        "min_roc_auc": float(
            trial.user_attrs[
                "min_roc_auc"
            ]
        ),
        "std_roc_auc": float(
            trial.user_attrs[
                "std_roc_auc"
            ]
        ),
        "params": dict(
            trial.params
        ),
    }

(
    RESULT_PATHS["optuna"]
    / "06_best_hyperparameters.json"
).write_text(
    json.dumps(
        best_hyperparameters_payload,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

display(
    model_selection_summary_df.sort_values(
        "mean_roc_auc",
        ascending=False,
    )
)
display(tree_model_ranking_df)

print(
    "Primary selected model:",
    PRIMARY_SELECTED_MODEL,
)
print(
    "Challenger model:",
    CHALLENGER_MODEL,
)
print(
    "Selected tree beats Logistic on mean ROC AUC:",
    selection_decision[
        "selected_tree_beats_logistic_on_mean_auc"
    ],
)


## 16. Freeze Stage 06 v3 manifest

In [ ]:
manifest = {
    "stage": 6,
    "status": (
        "frozen_after_integrity_checks"
    ),
    "stage_version": (
        "v3_fold_local_average_uniqueness"
    ),
    "notebook": (
        "06_optuna_model_selection.ipynb"
    ),
    "git_commit_sha": git_commit_sha(
        REPOSITORY_ROOT
    ),
    "software": software_manifest(),
    "study_signature": STUDY_SIGNATURE,
    "random_seed": SEED,
    "input_population": {
        "candidate_events": int(
            len(candidate_panel)
        ),
        "symbols": int(
            candidate_panel[
                "symbol"
            ].nunique()
        ),
        "features": MODEL_FEATURES,
        "numeric_features": (
            NUMERIC_FEATURES
        ),
        "categorical_features": (
            CATEGORICAL_FEATURES
        ),
        "stage04_primary_threshold_fraction": (
            0.15
        ),
    },
    "validation": {
        "source": (
            "frozen Stage 05 v2 folds"
        ),
        "stage05_code_commit_sha": (
            stage05_manifest[
                "git_commit_sha"
            ]
        ),
        "stage05_configuration_hash": (
            stage05_manifest[
                "configuration_hash"
            ]
        ),
        "fold_design_schema_version": (
            stage05_manifest[
                "fold_design_schema_version"
            ]
        ),
        "folds": int(len(folds)),
        "method": (
            "purged_anchored_walk_forward"
        ),
        "fold_weighting": "equal",
        "optuna_objective": (
            "mean_fold_roc_auc"
        ),
        "near_best_mean_auc_tolerance": (
            MEAN_AUC_TOLERANCE
        ),
        "diagnostic_threshold": (
            DIAGNOSTIC_THRESHOLD
        ),
        "threshold_selection_performed": (
            False
        ),
    },
    "sample_weighting": {
        "schema_version": (
            AVERAGE_UNIQUENESS_SCHEMA_VERSION
        ),
        "method": "average_uniqueness",
        "source_scope": (
            "current_fold_training_events_only"
        ),
        "concurrency_scope": (
            "within_symbol"
        ),
        "active_interval": (
            "inclusive_start_and_end"
        ),
        "normalization": (
            "fold_train_mean_one"
        ),
        "weighted_models": sorted(
            WEIGHTED_MODELS
        ),
        "dummy_prior_weighted": False,
        "validation_events_used": False,
        "validation_metrics_weighted": (
            False
        ),
        "return_attribution_multiplier_used": (
            False
        ),
        "sequential_bootstrap_used": False,
        "fold_audit": (
            average_uniqueness_audit_df.to_dict(
                orient="records"
            )
        ),
    },
    "optuna": {
        "required_complete_trials_per_model": (
            N_COMPLETE_TRIALS
        ),
        "pruner": "NopPruner",
        "every_trial_evaluates_all_folds": (
            True
        ),
        "study_names": study_names,
        "database_file": str(
            database_path
        ),
        "trial_state_counts": {
            model_name: trial_state_counts(
                study
            )
            for model_name, study in (
                studies.items()
            )
        },
    },
    "best_hyperparameters": (
        best_hyperparameters_payload
    ),
    "selection_decision": (
        selection_decision
    ),
    "unseen_test_used": False,
    "configuration_hash": stable_object_hash(
        {
            "optuna": optuna_config,
            "models": models_config,
            "feature_manifest": (
                feature_manifest_df.to_dict(
                    orient="records"
                )
            ),
            "fold_boundaries": (
                fold_boundaries_df.to_dict(
                    orient="records"
                )
            ),
            "average_uniqueness_audit": (
                average_uniqueness_audit_df.to_dict(
                    orient="records"
                )
            ),
        }
    ),
}

manifest_path = (
    RESULT_PATHS["manifests"]
    / "06_optuna_model_selection_manifest.json"
)

manifest_path.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("Manifest:", manifest_path)
print(
    "Manifest code SHA:",
    manifest["git_commit_sha"],
)


## 17. Final integrity checks

In [ ]:
assert candidate_errors_df.empty
assert calendar_errors_df.empty
assert fold_reproduction_audit_df[
    "exact_membership_match"
].all()

assert len(candidate_panel) == 118_464
assert len(MODEL_FEATURES) == 35
assert len(NUMERIC_FEATURES) == 34
assert CATEGORICAL_FEATURES == [
    "gmma_state"
]
assert len(folds) == 5

assert average_uniqueness_audit_df[
    "validation_events_used"
].eq(0).all()
assert average_uniqueness_audit_df[
    "nonfinite_raw_uniqueness"
].eq(0).all()
assert average_uniqueness_audit_df[
    "nonfinite_sample_weight"
].eq(0).all()
assert average_uniqueness_audit_df[
    "nonpositive_sample_weight"
].eq(0).all()
assert np.allclose(
    average_uniqueness_audit_df[
        "mean_sample_weight"
    ].to_numpy(dtype=float),
    1.0,
    atol=1.0e-12,
    rtol=1.0e-12,
)

expected_weight_rows = int(
    fold_reproduction_audit_df[
        "reproduced_train_events"
    ].sum()
)
assert len(
    all_fold_train_weights_df
) == expected_weight_rows

assert not all_fold_train_weights_df[
    [
        "fold_id",
        "event_id",
    ]
].duplicated().any()

assert (
    baseline_fold_metrics_df.loc[
        baseline_fold_metrics_df[
            "model_name"
        ].eq("dummy_prior"),
        "average_uniqueness_weighting_applied",
    ].eq(False).all()
)
assert (
    baseline_fold_metrics_df.loc[
        baseline_fold_metrics_df[
            "model_name"
        ].eq("logistic_regression"),
        "average_uniqueness_weighting_applied",
    ].eq(True).all()
)
assert tuned_fold_metrics_df[
    "average_uniqueness_weighting_applied"
].eq(True).all()
assert all_fold_metrics_df[
    "validation_metrics_weighted"
].eq(False).all()

assert all_fold_metrics_df.groupby(
    "model_name"
)["fold_id"].nunique().eq(5).all()

assert set(
    all_fold_metrics_df[
        "model_name"
    ]
) == {
    "dummy_prior",
    "logistic_regression",
    "random_forest",
    "xgboost",
}

assert all_fold_metrics_df[
    [
        "roc_auc",
        "average_precision",
        "log_loss",
        "brier_score",
        "balanced_accuracy",
        "specificity",
        "sensitivity",
        "precision",
        "f1",
        "mcc",
    ]
].notna().all().all()

assert all_predictions_df[
    "probability_positive"
].between(0.0, 1.0).all()
assert all_predictions_df[
    "validation_metric_weight"
].eq(1.0).all()

assert not all_predictions_df[
    [
        "model_name",
        "fold_id",
        "event_id",
    ]
].duplicated().any()

for model_name, study in (
    studies.items()
):
    counts = trial_state_counts(study)

    assert counts["COMPLETE"] >= (
        N_COMPLETE_TRIALS
    )
    assert counts["PRUNED"] == 0
    assert counts["FAIL"] == 0

    for trial in study.trials:
        if trial.state == (
            optuna.trial.TrialState.COMPLETE
        ):
            assert trial.user_attrs[
                "folds_evaluated"
            ] == 5

assert isinstance(
    studies["random_forest"].pruner,
    optuna.pruners.NopPruner,
)
assert isinstance(
    studies["xgboost"].pruner,
    optuna.pruners.NopPruner,
)
assert (
    studies["random_forest"].sampler
    is not studies["xgboost"].sampler
)

assert int(
    models_config["random_forest"][
        "search_space"
    ]["n_estimators"]["high"]
) == 150
assert int(
    models_config["random_forest"][
        "search_space"
    ]["max_depth"]["high"]
) == 15
assert int(
    models_config["xgboost"][
        "search_space"
    ]["n_estimators"]["high"]
) == 150
assert int(
    models_config["xgboost"][
        "search_space"
    ]["max_depth"]["high"]
) == 15

for model_name, trial in (
    selected_trials.items()
):
    ranking = all_trial_rankings_df.loc[
        all_trial_rankings_df[
            "model_name"
        ].eq(model_name)
    ]
    selected_row = ranking.loc[
        ranking[
            "selected_by_hierarchy"
        ]
    ].iloc[0]

    assert int(
        selected_row[
            "trial_number"
        ]
    ) == int(trial.number)
    assert selected_row[
        "inside_near_best_mean_auc_band"
    ]

assert PRIMARY_SELECTED_MODEL in {
    "random_forest",
    "xgboost",
}
assert CHALLENGER_MODEL in {
    "random_forest",
    "xgboost",
}
assert (
    PRIMARY_SELECTED_MODEL
    != CHALLENGER_MODEL
)

assert selection_decision[
    "threshold_selected"
] is False
assert selection_decision[
    "unseen_test_used"
] is False
assert manifest[
    "unseen_test_used"
] is False

print("Notebook 06 integrity checks: PASSED")
print(
    "Candidate events:",
    len(candidate_panel),
)
print(
    "Final pooled-model features:",
    len(MODEL_FEATURES),
)
print(
    "Frozen Stage 05 v2 folds:",
    len(folds),
)
print(
    "Average Uniqueness weighting:",
    "fold-local within-symbol",
)
print(
    "Validation events used in weighting: 0"
)
print(
    "Weighted models:",
    sorted(WEIGHTED_MODELS),
)
print("Dummy prior weighted: False")
print("Validation metrics weighted: False")
print(
    "COMPLETE trials per tuned model:",
    N_COMPLETE_TRIALS,
)
print("Pruning enabled: False")
print(
    "Maximum n_estimators for RF/XGB: 150"
)
print(
    "Maximum max_depth for RF/XGB: 15"
)
print(
    "Primary objective:",
    "equal-fold mean ROC AUC",
)
print(
    "Near-best mean-AUC tolerance:",
    MEAN_AUC_TOLERANCE,
)
print(
    "Hierarchical tie-break:",
    "worst-fold AUC, then fold AUC std",
)
print(
    "Threshold selection performed: False"
)
print(
    "Primary selected model:",
    PRIMARY_SELECTED_MODEL,
)
print(
    "Challenger model:",
    CHALLENGER_MODEL,
)
print(
    "Unseen-test used in Stage 06 decisions: False"
)
print(
    "Next stage: Notebook 07 — Frozen model training "
    "on the complete eligible train-only candidate population."
)


## Review before freezing Stage 06

Inspect:

- `results/audits/06_average_uniqueness_audit.csv`
- `results/audits/06_average_uniqueness_symbol_audit.csv`
- `results/folds/06_fold_train_average_uniqueness_weights.csv`
- `results/optuna/06_random_forest_trials.csv`
- `results/optuna/06_xgboost_trials.csv`
- `results/optuna/06_random_forest_trial_selection_ranking.csv`
- `results/optuna/06_xgboost_trial_selection_ranking.csv`
- `results/optuna/06_best_hyperparameters.json`
- `results/metrics/06_model_selection_fold_metrics.csv`
- `results/metrics/06_model_selection_summary.csv`
- `results/metrics/06_tree_model_hierarchical_ranking.csv`
- `results/predictions/06_walk_forward_validation_predictions.csv`
- `results/manifests/06_model_selection_decision.json`
- `results/manifests/06_optuna_model_selection_manifest.json`
- `results/audits/06_fold_reproduction_audit.csv`
- `results/audits/06_candidate_panel_audit.csv`

Notebook 07 must consume the frozen Stage 06 hyperparameters and selection
decision. It must not rerun Optuna after observing unseen-test performance.
